In [41]:
import os
import time
import glob
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.chrome.service import Service
import requests
import pandas as pd
import urllib.parse
import threading
from time import sleep
from threading import Thread
import glob
import shutil
import csv
from pymongo import MongoClient
import os
from pymongo.errors import DuplicateKeyError
from typing import List, Dict, Optional, Union
from pymongo import MongoClient, UpdateOne
from pymongo.errors import BulkWriteError
from datetime import datetime

In [42]:
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project-2' 
mongoDB_collection = 'sdk_information'

In [43]:
def build_driver():
    chrome_options = Options()
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--disable-popup-blocking")
    chrome_options.add_argument('--ignore-ssl-errors=yes')
    chrome_options.add_argument('--ignore-certificate-errors')
    chrome_options.add_argument('--allow-running-insecure-content')
    chrome_options.add_argument('--no-proxy-server')
    chrome_options.add_argument("--no-first-run")
    chrome_options.add_argument("--no-default-browser-check")
    chrome_options.add_argument("--disable-search-engine-choice-screen")
    # CHANGE (ổn định thêm trên Windows/Chrome mới):
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--remote-allow-origins=*")

    prefs = {
        'profile.default_content_setting_values.automatic_downloads': 1,
        'profile.default_content_setting_values.popups': 0,
        'profile.default_content_setting_values.notifications': 2,
        "profile.default_content_setting_values.geolocation": 2,
        "profile.default_content_setting_values.media_stream": 2,
        "profile.default_content_setting_values.cookies": 2,
        "profile.default_content_setting_values.javascript": 1,
        "profile.default_content_setting_values.plugins": 1,
        "profile.default_content_setting_values.fullscreen": 2,
        'profile.default_content_settings.popups': 0,
        'profile.default_content_setting_values.default_search_engine_id': 1,
        'profile.default_content_setting_values.prompt_for_download': False,
        'profile': {'default_content_setting_values': {'search_suggest_enabled': False}},
        'intl.accept_languages': 'en,en_US',
        'first_run': False,
        'browser.startup.page': 0,
        'browser.startup.homepage': 'about:blank',
    }
    chrome_options.add_experimental_option("prefs", prefs)

    # CHỌN 1 TRONG 2 CÁCH KHỞI TẠO:
    # Cách A: dùng chromedriver.exe sẵn có
    service = Service(executable_path='chromedriver.exe')
    driver = webdriver.Chrome(service=service, options=chrome_options)

    # Cách B (khuyên dùng): tự tải đúng version
    # driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

    driver.maximize_window()
    return driver
# CHANGE: helper điều hướng có retry & tự recreate driver khi connection chết
def navigate_with_retry(current_driver, url, max_retries=1):
    for attempt in range(max_retries + 1):
        try:
            current_driver.get(url)
            return current_driver  # ok
        except WebDriverException as e:
            msg = str(e)
            # các chuỗi báo hiệu driver server đã chết
            if ("Max retries exceeded" in msg or
                "Failed to establish a new connection" in msg or
                "invalid session id" in msg or
                "disconnected" in msg):
                print(f"[navigate_with_retry] Driver connection died on attempt {attempt}. Recreating...")
                try:
                    current_driver.quit()
                except Exception:
                    pass
                # recreate driver
                new_driver = build_driver()
                # thử lại ngay
                if attempt < max_retries:
                    continue
                else:
                    # lần cuối vẫn fail thì raise
                    new_driver.get(url)  # nếu lỗi sẽ raise ra ngoài
                    return new_driver
            else:
                # lỗi khác -> ném ra ngoài
                raise
    return current_driver
def connect_mongodb(uri, db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def insert_one_json(collection, data: dict) -> bool:
    """
    Chèn 1 document JSON vào MongoDB.
    Trả về True nếu chèn thành công, False nếu lỗi.
    """
    try:
        collection.insert_one(dict(data))  # copy để tránh thay đổi data gốc
        return True
    except Exception as e:
        print(f"[Insert Error] {e}")
        return False

def create_sdk_json(sdk_name, maven_id, registration, sdk_category):
    """
    Tạo JSON data để chèn vào MongoDB (tất cả các trường đều bắt buộc).
    """
    return {
        "sdk_name": sdk_name.strip(),
        "maven_id": maven_id.strip(),
        "registration": registration.strip(),
        "sdk_category": sdk_category.strip()
    }


In [44]:
array_sdk_category = ["ads","analytics","data-management","location",
                      "marketing-and-engagement","payments","social","user-authentication","user-support"]
mongodb_connection = connect_mongodb(mongoDB_uri, mongoDB_database, mongoDB_collection)

In [45]:
try:
    # 1.Get list app download link
    # 1.1 Setup selenium
    # CHANGE: dùng hàm build_driver()
    web = build_driver()
    wait = WebDriverWait(web, 20)

    for i in range(len(array_sdk_category)):
        print("----------------------------- Loop-"+str(i)+" -----------------------------")
        sdk_category_name = array_sdk_category[i]
        print("sdk_category_name", sdk_category_name)
        sdk_path = "https://play.google.com/sdks/categories/" + sdk_category_name
        print("sdk_path: ", sdk_path)

        try:
            # CHANGE: dùng navigate_with_retry() thay vì web.get() trực tiếp
            web = navigate_with_retry(web, sdk_path, max_retries=1)
            time.sleep(1.5)

            # Đợi toàn bộ các dòng (row) xuất hiện
            rows = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div.particle-table-row[role='row']")
                )
            )

            sdk_data_array = []

            def safe_get_text(scope_el, by, sel):
                try:
                    return scope_el.find_element(by, sel).text.strip()
                except Exception:
                    return ""

            for r in rows:
                sdk_name = safe_get_text(
                    r,
                    By.XPATH,
                    ".//ess-cell[@essfield='sdk-name-column']//div[contains(@class,'line')]"
                )
                # CHANGE: làm sạch tên bị lỗi prefix ":" (như ': Tappx Android SDK')
                if sdk_name:
                    sdk_name = sdk_name.lstrip(": ").strip()

                maven_id = safe_get_text(
                    r,
                    By.XPATH,
                    ".//ess-cell[@essfield='maven-id-column']//span[contains(@class,'break-long-words')]"
                )

                registration = safe_get_text(
                    r,
                    By.XPATH,
                    ".//ess-cell[@essfield='play-registered-column']//span[contains(@class,'main-text')]"
                )

                if any([sdk_name, maven_id, registration]):
                    sdk_data_array.append({
                        "sdk_name": sdk_name,
                        "maven_id": maven_id,
                        "registration": registration
                    })

            # In & insert
            for sdk in sdk_data_array:
                sdk_data_insert = create_sdk_json(
                    sdk['sdk_name'],
                    sdk['maven_id'],
                    sdk['registration'],
                    sdk_category_name
                )
                print(sdk_data_insert)
                insert_stt = insert_one_json(mongodb_connection, sdk_data_insert)
                print("Insert stt: ", insert_stt)

            time.sleep(0.5)

        except Exception as e:
            print(f"[Loop-{i}] ERROR: {e}")
            # tiếp tục category tiếp theo
            continue

    # CHANGE: đóng driver sau khi hoàn tất
    web.close()

except KeyboardInterrupt:
    try:
        web.close()
    except Exception:
        pass

----------------------------- Loop-0 -----------------------------
sdk_category_name ads
sdk_path:  https://play.google.com/sdks/categories/ads
{'sdk_name': 'Liftoff Monetize - Vungle Android SDK v7', 'maven_id': 'com.vungle:vungle-ads', 'registration': 'Registered in Google Play SDK Console', 'sdk_category': 'ads'}
Insert stt:  True
{'sdk_name': 'Tappx Android SDK', 'maven_id': 'com.tappx.sdk.android:tappx-sdk', 'registration': 'Registered in Google Play SDK Console', 'sdk_category': 'ads'}
Insert stt:  True
{'sdk_name': 'APS AdMob Adapter', 'maven_id': 'com.amazon.android:aps-admob-adapter', 'registration': '', 'sdk_category': 'ads'}
Insert stt:  True
{'sdk_name': 'APS Android SDK', 'maven_id': 'com.amazon.android:aps-sdk', 'registration': '', 'sdk_category': 'ads'}
Insert stt:  True
{'sdk_name': 'Ad Kit', 'maven_id': 'com.snap.adkit:adkit', 'registration': 'Registered in Google Play SDK Console', 'sdk_category': 'ads'}
Insert stt:  True
{'sdk_name': 'Ad Quality SDK', 'maven_id': 'co